In [16]:
!pip install torchmetrics

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from sklearn.preprocessing import StandardScaler
from torch.utils.tensorboard import SummaryWriter
import matplotlib.pyplot as plt
import random
import time

In [18]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/organizations/uciml/forest-cover-type-dataset/covtype.csv


In [19]:
import pandas as pd

df = pd.read_csv("/kaggle/input/datasets/organizations/uciml/forest-cover-type-dataset/covtype.csv")

df.head()

,Elevation,Aspect,Slope,Horizontal_Distance_To_Hydrology,Vertical_Distance_To_Hydrology,Horizontal_Distance_To_Roadways,Hillshade_9am,Hillshade_Noon,Hillshade_3pm,Horizontal_Distance_To_Fire_Points,...,Soil_Type32,Soil_Type33,Soil_Type34,Soil_Type35,Soil_Type36,Soil_Type37,Soil_Type38,Soil_Type39,Soil_Type40,Cover_Type
0,2596,51,3,258,0,510,221,232,148,6279,...,0,0,0,0,0,0,0,0,0,5
1,2590,56,2,212,-6,390,220,235,151,6225,...,0,0,0,0,0,0,0,0,0,5
2,2804,139,9,268,65,3180,234,238,135,6121,...,0,0,0,0,0,0,0,0,0,2
3,2785,155,18,242,118,3090,238,238,122,6211,...,0,0,0,0,0,0,0,0,0,2
4,2595,45,2,153,-1,391,220,234,150,6172,...,0,0,0,0,0,0,0,0,0,5


In [20]:
print("Shape:", df.shape)
print("Classes:", df["Cover_Type"].unique())

Shape: (581012, 55)
Classes: [5 2 1 7 3 6 4]


In [21]:
X = df.drop("Cover_Type", axis=1).values
y = df["Cover_Type"].values - 1   # Make classes 0–6

scaler = StandardScaler()
X = scaler.fit_transform(X)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (581012, 54)
y shape: (581012,)


In [26]:
class ForestDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [27]:
class MLP(nn.Module):
    def __init__(self, input_dim=54, hidden1=256, hidden2=128, output_dim=7, dropout=0.3):
        super(MLP, self).__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden1),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden1, hidden2),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden2, output_dim)
        )

    def forward(self, x):
        return self.net(x)

In [28]:
def train_model(seed):

    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.3, random_state=seed)

    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.5, random_state=seed)

    train_dataset = ForestDataset(X_train, y_train)
    val_dataset = ForestDataset(X_val, y_val)
    test_dataset = ForestDataset(X_test, y_test)

    train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=1024)
    test_loader = DataLoader(test_dataset, batch_size=1024)

    model = MLP()
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    writer = SummaryWriter(log_dir=f"runs/seed_{seed}")

    best_val_acc = 0

    for epoch in range(10):

        model.train()
        total_loss = 0

        for xb, yb in train_loader:
            optimizer.zero_grad()
            outputs = model(xb)
            loss = criterion(outputs, yb)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        # Validation
        model.eval()
        val_preds = []
        val_true = []

        with torch.no_grad():
            for xb, yb in val_loader:
                outputs = model(xb)
                preds = torch.argmax(outputs, dim=1)
                val_preds.extend(preds.numpy())
                val_true.extend(yb.numpy())

        val_acc = accuracy_score(val_true, val_preds)

        writer.add_scalar("Loss/train", total_loss, epoch)
        writer.add_scalar("Accuracy/val", val_acc, epoch)

        print(f"Seed {seed} | Epoch {epoch+1} | Loss {total_loss:.4f} | Val Acc {val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), f"best_model_seed_{seed}.pt")

    writer.close()

    return best_val_acc

In [29]:
results = []

start_time = time.time()

for seed in range(30):
    acc = train_model(seed)
    results.append(acc)

print("Finished 30 trials")

Seed 0 | Epoch 1 | Loss 284.4488 | Val Acc 0.7510
Seed 0 | Epoch 2 | Loss 232.6850 | Val Acc 0.7726
Seed 0 | Epoch 3 | Loss 217.2825 | Val Acc 0.7861
Seed 0 | Epoch 4 | Loss 207.1101 | Val Acc 0.7995
Seed 0 | Epoch 5 | Loss 199.1495 | Val Acc 0.8047
Seed 0 | Epoch 6 | Loss 192.9595 | Val Acc 0.8169
Seed 0 | Epoch 7 | Loss 188.2768 | Val Acc 0.8252
Seed 0 | Epoch 8 | Loss 183.8803 | Val Acc 0.8313
Seed 0 | Epoch 9 | Loss 180.4657 | Val Acc 0.8340
Seed 0 | Epoch 10 | Loss 177.3109 | Val Acc 0.8382
Seed 1 | Epoch 1 | Loss 289.6884 | Val Acc 0.7489
Seed 1 | Epoch 2 | Loss 234.0438 | Val Acc 0.7714
Seed 1 | Epoch 3 | Loss 218.0303 | Val Acc 0.7849
Seed 1 | Epoch 4 | Loss 207.0000 | Val Acc 0.8005
Seed 1 | Epoch 5 | Loss 199.2023 | Val Acc 0.8096
Seed 1 | Epoch 6 | Loss 192.8693 | Val Acc 0.8169
Seed 1 | Epoch 7 | Loss 187.5735 | Val Acc 0.8232
Seed 1 | Epoch 8 | Loss 182.8471 | Val Acc 0.8290
Seed 1 | Epoch 9 | Loss 179.3489 | Val Acc 0.8351
Seed 1 | Epoch 10 | Loss 175.8211 | Val Acc 0.838

In [36]:
results = np.array(results)

print("Mean Accuracy:", results.mean())
print("Std Accuracy:", results.std())
print("Best Accuracy:", results.max())
print("Worst Accuracy:", results.min())

Mean Accuracy: 0.8390394559696468
Std Accuracy: 0.0016475550317353162
Best Accuracy: 0.8416903800257022
Worst Accuracy: 0.8338190747200294


In [39]:
%load_ext tensorboard
%tensorboard --logdir runs

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6006 (pid 195), started 0:00:41 ago. (Use '!kill 195' to kill it.)

<IPython.core.display.Javascript object>

In [40]:
!kill 195


In [41]:
%load_ext tensorboard
%tensorboard --logdir /kaggle/working/runs

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


<IPython.core.display.Javascript object>

In [42]:
best_seed = np.argmax(results)
print("Best Seed:", best_seed)

model = MLP()
model.load_state_dict(torch.load(f"best_model_seed_{best_seed}.pt"))
model.eval()

# Test split again
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=best_seed)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=best_seed)

test_dataset = ForestDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=1024)

all_preds = []
all_true = []

with torch.no_grad():
    for xb, yb in test_loader:
        outputs = model(xb)
        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.numpy())
        all_true.extend(yb.numpy())

cm = confusion_matrix(all_true, all_preds)
print(cm)

Best Seed: 5
[[25819  5326     0     0    22     4   416]
 [ 3978 37777   325     0   235   222    35]
 [    2   317  4856    24     2   288     0]
 [    0     0   172   208     0    26     0]
 [   82   664    48     0   630     8     0]
 [    6   289   930     6     1  1369     0]
 [  402    27     0     0     0     0  2636]]


In [43]:
precision, recall, f1, _ = precision_recall_fscore_support(
    all_true, all_preds, average="macro")

print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

Precision: 0.8028756170257545
Recall: 0.7040095741293548
F1 Score: 0.7394986378151994


In [44]:
import os

print(os.getcwd())
print(os.listdir("/kaggle/working"))

/kaggle/working
['best_model_seed_25.pt', 'best_model_seed_7.pt', 'best_model_seed_21.pt', 'best_model_seed_28.pt', 'best_model_seed_10.pt', 'best_model_seed_23.pt', 'best_model_seed_29.pt', 'best_model_seed_11.pt', '.virtual_documents', 'best_model_seed_27.pt', 'best_model_seed_26.pt', 'best_model_seed_1.pt', 'best_model_seed_15.pt', 'best_model_seed_17.pt', 'best_model_seed_5.pt', 'best_model_seed_0.pt', 'best_model_seed_8.pt', 'best_model_seed_24.pt', 'best_model_seed_16.pt', 'best_model_seed_22.pt', 'best_model_seed_6.pt', 'best_model_seed_4.pt', 'runs', 'best_model_seed_19.pt', 'best_model_seed_18.pt', 'best_model_seed_20.pt', 'best_model_seed_13.pt', 'best_model_seed_2.pt', 'best_model_seed_12.pt', 'best_model_seed_3.pt', 'best_model_seed_14.pt', 'best_model_seed_9.pt']


In [50]:
!ls /kaggle/working/runs

seed_0	 seed_12  seed_16  seed_2   seed_23  seed_27  seed_4  seed_8
seed_1	 seed_13  seed_17  seed_20  seed_24  seed_28  seed_5  seed_9
seed_10  seed_14  seed_18  seed_21  seed_25  seed_29  seed_6
seed_11  seed_15  seed_19  seed_22  seed_26  seed_3   seed_7


In [63]:
import os
print(os.listdir("runs"))

['seed_26', 'seed_8', 'seed_27', 'seed_20', 'seed_17', 'seed_9', 'seed_13', 'seed_14', 'seed_25', 'seed_6', 'seed_12', 'seed_2', 'seed_10', 'seed_3', 'seed_15', 'seed_16', 'seed_29', 'seed_0', 'seed_22', 'seed_23', 'seed_18', 'seed_19', 'seed_28', 'seed_5', 'seed_1', 'seed_24', 'seed_11', 'seed_21', 'seed_4', 'seed_7']


In [70]:
torch.save(model.state_dict(), "/kaggle/working/best_model.pt")